In [54]:
import pandas as pd
import torch

In [55]:
train_df = pd.read_csv("/content/train.csv")

Q1. Label Encoding

Convert the answer column in train.csv into numeric labels using the following mapping:

A = 0

B = 1

C = 2

D = 3

E = 4

What is the encoded numeric label for the row at index 150?

In [56]:
label_map = {"A": 0 , "B": 1, "C":2, "D":3, "E":4}

train_df["answer"] = train_df["answer"].apply(lambda answer: label_map[answer])

In [57]:
print(f"Encoded numeric label for row at index 150 is: {train_df.iloc[150]["answer"]}")

Encoded numeric label for row at index 150 is: 2


Q2. Prompt-Option Formatting

For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?
*

In [58]:
prompt_0 = train_df.iloc[0]["prompt"]
optionB_0 = train_df.iloc[0]["B"]

optionB = str(prompt_0) + "[SEP]" + str(optionB_0)

print(optionB)

print(f'Exact Character length of this formatted input string:{len(optionB)}')

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.[SEP]Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Exact Character length of this formatted input string:405


Q3. Single-Row MCQ Tokenization

Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:

padding = "max_length"

truncation = True

max_length = 128

return_tensors = "pt"


After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [59]:
from transformers import AutoTokenizer
MODEL = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [60]:
options = torch.randn(1,5,128)

for option in ["A", "B", "C", "D", "E"]:
  formatted_option = str(prompt_0) + "[SEP]" + str(train_df.iloc[0][option])

  tokenized_input = tokenizer(
      formatted_option,
      padding = "max_length",
      truncation = True,
      max_length = 128,
      return_tensors = "pt"
  )
  options[0,label_map[option]]= tokenized_input["input_ids"]

print(options.shape)

torch.Size([1, 5, 128])


Q4. Batch MCQ Tokenization

Tokenize the first 16 rows of train.csv as multiple-choice examples.

Each row has 5 choices.

Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [81]:
num = 16
input = []

for i in range(num):
  row = train_df.iloc[i]

  prompt = row["prompt"]
  options= []
  for item in label_map.keys():
    options.append(row[item])

  tokenized_input = tokenizer(
    options,
    padding = "max_length",
    truncation = True,
    max_length = 128,
    return_tensors = "pt"
  )

  input.append(tokenized_input["input_ids"])


input_tokens = torch.stack(input, dim =0)

input_tokens_shape = input_tokens.shape

print(f"Total token positions in this tensor: {input_tokens_shape[0]*input_tokens_shape[1]*input_tokens_shape[2]}")

Total token positions in this tensor: 10240


Q5. Multiple-Choice Logits

Load bert-base-uncased using AutoModelForMultipleChoice.

Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?